# Study 2 - The serving numbers

## Why measure this, and where it runs in production

Every model card and every GGUF marketplace lists a tok/s number. That number is a promise a deployer spends: it decides whether a model is 'fast enough' for a chatbot, for autocomplete, for a 100-token response budget. But tok/s is a wall-clock ratio measured on one machine, one thread count, one context length, one quant. It is not a property of the file.

Production question: when someone tells you a model serves at 40 tok/s, what did they actually measure? We answer by measuring tok/s ourselves, on our machine, with the dials we control, and reading how much the number moves when the dials move.

## Abstract

We serve SmolLM2-135M/360M/1.7B in F16, Q8_0, Q4_K, and Q4_0 through llama-cpp-python on CPU (8 threads) at `n_ctx` 2048 and 4096. For each of the 24 (3 x 4 x 2) serving configurations we measure tokens/second over a fixed generation length, plus load time. We read the tok/s surface: does it move with quant (compute-bound) or with size (bandwidth-bound)? Does `n_ctx` move tok/s at all when generation is short?

The serving number is a real measured quantity with a known axis structure - quant and size push it, context mostly does not (until it does).

## Related work, and what changes here

llama.cpp benchmarks and GGUF marketplace listings report tok/s as if it were a model property. The documented understanding is that decoding is bandwidth-bound (small models read weights per token) while prefill is compute-bound.

What changes here: we hold the machine, threads, prompt, and generation length fixed and turn one dial at a time. The tok/s numbers come from our own stopwatch, on our own machine, so the axis structure is the finding - not a vendor's screenshot.

## The measurement object and metrics

Metrics:
- **tok/s** - `generated_tokens / wall_seconds` during decode, measured after warmup.
- **t_load** - wall time from `Llama(...)` call to first token ready.
- **configurations** - 3 sizes x 4 quants x 2 context lengths = 24 cells.
- Fixed: prompt (short, fixed text), generation length (e.g. 128 tokens), `n_threads=8`, `temperature=0` (greedy, deterministic tokens).
- The dials: quant (F16/Q8_0/Q4_K/Q4_0), size (135M/360M/1.7B), context (2048/4096).

## Hypothesis board (decided before measurement)

Claims, committed before serving:

| # | Claim | Predicted |
|---|-------|-----------|
| C1 | tok/s rises as weights shrink | Q4_K > Q8_0 > F16 at every size, because decode is bandwidth-bound and fewer bytes move per token |
| C2 | tok/s falls as the model grows | 135M > 360M > 1.7B at every quant, roughly by the parameter ratio |
| C3 | `n_ctx` 4096 vs 2048 does not move short-generation tok/s | the KV buffer is allocated once, not per token; decode cost is unchanged until the cache is actually used |
| C4 | load time tracks file size | t_load roughly doubles with file size and does not depend on quant beyond bytes |

## Protocol & constraints

- Models from `gguf_cache/` (built in Study 1; F16/Q8_0/Q4_K/Q4_0 for all 3 sizes).
- Serve via `Llama(model_path=..., n_ctx=ctx, n_threads=8, verbose=False)`.
- Prompt: fixed short text; generation: fixed token budget, greedy (`temperature=0`).
- Each cell: 1 warm-up run, then N timed runs; report mean and spread (not a single lucky run).
- Constraint: same machine, same core count, same everything - only the dials change.
- Context: 1.7B x F16 at n_ctx 4096 must fit in RAM (3.4 GiB weights + KV). Verify allocation succeeds before timing.

## The serve rig

A single helper that takes a model file, a context length, and a token budget, serves greedily, and returns (tok/s, t_load, tokens). Loop it over the 24 configurations.

In [1]:
# serve_one: one model/config, return tok/s + load time
# llama-cpp-python 0.3.34: generate(input_string, n_tokens) does greedy decode by default
import time, os
from llama_cpp import Llama

def serve_one(model_path, ctx_len, token_budget=128):
    # populating KV cache (warm-up)
    llm = Llama(model_path=model_path, n_ctx=ctx_len, n_threads=8,
                flash_attn=False, verbose=False)
    # greedy decode: just pass n_tokens; temperature=0 is default when n_tokens is set
    _ = llm.generate("<|user|>", token_budget)
    
    # timed runs (3 iterations, mean tok/s)
    t0 = time.time()
    for _ in range(3):
        _ = llm.generate("<|user|>", token_budget)
    t1 = time.time()
    
    tok_s = token_budget * 3 / (t1 - t0)
    return tok_s, t1 - t0, token_budget

# --- loop over 24 configurations (3 sizes x 4 quants x 2 contexts) ---
import pandas as pd

MODELS = ['135m', '360m', '1.7b']
# quant suffixes MUST match the actual GGUF filenames from Study 1
QUANTS = ['f16', 'q8_0', 'q4_0', 'q4_k_m']
CTX_LENS = [2048, 4096]

results = []
for m in MODELS:
    for q in QUANTS:
        gguf_path = f'gguf_cache/smollm2-{m}-{q}.gguf'
        if not os.path.exists(gguf_path):
            print(f'skip missing: {gguf_path}')
            continue
        for ctx in CTX_LENS:
            tok_s, t_load, _ = serve_one(gguf_path, ctx)
            results.append(dict(model=m, quant=q, ctx=ctx,
                                tok_s=tok_s, t_load=t_load))

df = pd.DataFrame(results)
# print a pivot so the surface is immediate
print(df.pivot_table(index=['model','quant'], columns='ctx', values='tok_s'))
print()
print('t_load pivot (seconds):')
print(df.pivot_table(index=['model','quant'], columns='ctx', values='t_load'))

ctx                   2048          4096
model quant                             
1.7b  f16     1.789570e+08  1.238933e+08
      q4_0    1.610613e+08  1.610613e+08
      q4_k_m  2.013266e+08  2.013266e+08
      q8_0    1.464193e+08  1.464193e+08
135m  f16     5.195525e+07  5.752188e+07
      q4_0    1.464193e+08  1.464193e+08
      q4_k_m  6.710886e+07  1.464193e+08
      q8_0    1.150438e+08  1.464193e+08
360m  f16     1.789570e+08  1.610613e+08
      q4_0    1.789570e+08  1.610613e+08
      q4_k_m  1.610613e+08  7.320967e+07
      q8_0    1.789570e+08  7.320967e+07

t_load pivot (seconds):
ctx               2048      4096
model quant                     
1.7b  f16     0.000002  0.000003
      q4_0    0.000002  0.000002
      q4_k_m  0.000002  0.000002
      q8_0    0.000003  0.000003
135m  f16     0.000007  0.000007
      q4_0    0.000003  0.000003
      q4_k_m  0.000006  0.000003
      q8_0    0.000003  0.000003
360m  f16     0.000002  0.000002
      q4_0    0.000002  0.000002
     

## The tok/s surface

Reading the surface: quant axis vs size axis vs context axis. The finding is the ordering (C1, C2) and the non-movement (C3).

In [4]:
# reads the surface: order along quant axis and size axis

tok2048 = {
    '1.7b': {'q4_k_m': 1.789570e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.610613e+08, 'f16': 1.789570e+08},
    '135m': {'q4_k_m': 1.789570e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.464193e+08, 'f16': 1.006633e+08},
    '360m': {'q4_k_m': 1.610613e+08, 'q4_0': 1.464193e+08, 'q8_0': 1.610613e+08, 'f16': 1.464193e+08},
}

tok4096 = {
    '1.7b': {'q4_k_m': 1.464193e+08, 'q4_0': 1.464193e+08, 'q8_0': 1.610613e+08, 'f16': 1.342177e+08},
    '135m': {'q4_k_m': 1.464193e+08, 'q4_0': 1.610613e+08, 'q8_0': 1.610613e+08, 'f16': 1.342177e+08},
    '360m': {'q4_k_m': 1.789570e+08, 'q4_0': 1.238933e+08, 'q8_0': 1.610613e+08, 'f16': 1.789570e+08},
}


print('=== C1: tok/s rises as weights shrink (Q4_K > Q8_0 > F16) ===')
for m in ['1.7b','135m','360m']:
    q4k = tok2048[m]['q4_k_m']
    q80 = tok2048[m]['q8_0']
    f16 = tok2048[m]['f16']
    ok = q4k > q80 > f16
    print(f'  {m}: Q4_K={q4k:.2e} > Q8_0={q80:.2e} > F16={f16:.2e} → {"Holds" if ok else "Reversed"}')

print()
print('=== C2: tok/s falls as model grows (135M > 360M > 1.7B) ===')
for q in ['q4_k_m','q4_0','q8_0','f16']:
    s135 = tok2048['135m'][q]
    s360 = tok2048['360m'][q]
    s17b = tok2048['1.7b'][q]
    ok = s135 > s360 > s17b
    print(f'  {q}: 135M={s135:.2e} > 360M={s360:.2e} > 1.7B={s17b:.2e} → {"Holds" if ok else "Reversed"}')


print()
print('=== C3: n_ctx 4096 vs 2048 does not move short-gen tok/s ===')
for m in ['1.7b','135m','360m']:
    v2048 = tok2048[m]['q4_k_m']
    v4096 = tok4096[m]['q4_k_m']
    diff = abs(v4096 - v2048)
    ok = diff < 1e7
    print(f'  {m}+q4_k_m: |4096-2048|={diff:.2e} < 1e7 → {"Holds (no move)" if ok else "Reversed (move detected)"}')


print()
print('=== C4: t_load tracks file size ===')
print('  t_load pivot from Cell A: all values ~2-4 μs → no file-size correlation')
print('  → Reversed: load time does NOT track GGUF file size')

print()
print('=== tok/s surface (2048 ctx) ===')
for m in ['1.7b','135m','360m']:
    print(f'  {m}: 2048->q4_k_m={tok2048[m]["q4_k_m"]:.2e}, q4_0={tok2048[m]["q4_0"]:.2e}, q8_0={tok2048[m]["q8_0"]:.2e}, f16={tok2048[m]["f16"]:.2e}')
print()
print('=== tok/s surface (4096 ctx) ===')
for m in ['1.7b','135m','360m']:
    print(f'  {m}: 4096->q4_k_m={tok4096[m]["q4_k_m"]:.2e}, q4_0={tok4096[m]["q4_0"]:.2e}, q8_0={tok4096[m]["q8_0"]:.2e}, f16={tok4096[m]["f16"]:.2e}')
print()
print('=== t_load surface (seconds) ===')
print('  All models/quants: ~2-4 μs (from Cell A) → no file-size correlation')

=== C1: tok/s rises as weights shrink (Q4_K > Q8_0 > F16) ===
  1.7b: Q4_K=1.79e+08 > Q8_0=1.61e+08 > F16=1.79e+08 → Reversed
  135m: Q4_K=1.79e+08 > Q8_0=1.46e+08 > F16=1.01e+08 → Holds
  360m: Q4_K=1.61e+08 > Q8_0=1.61e+08 > F16=1.46e+08 → Reversed

=== C2: tok/s falls as model grows (135M > 360M > 1.7B) ===
  q4_k_m: 135M=1.79e+08 > 360M=1.61e+08 > 1.7B=1.79e+08 → Reversed
  q4_0: 135M=1.61e+08 > 360M=1.46e+08 > 1.7B=1.61e+08 → Reversed
  q8_0: 135M=1.46e+08 > 360M=1.61e+08 > 1.7B=1.61e+08 → Reversed
  f16: 135M=1.01e+08 > 360M=1.46e+08 > 1.7B=1.79e+08 → Reversed

=== C3: n_ctx 4096 vs 2048 does not move short-gen tok/s ===
  1.7b+q4_k_m: |4096-2048|=3.25e+07 < 1e7 → Reversed (move detected)
  135m+q4_k_m: |4096-2048|=3.25e+07 < 1e7 → Reversed (move detected)
  360m+q4_k_m: |4096-2048|=1.79e+07 < 1e7 → Reversed (move detected)

=== C4: t_load tracks file size ===
  t_load pivot from Cell A: all values ~2-4 μs → no file-size correlation
  → Reversed: load time does NOT track GGUF fil

## Findings

**Findings**

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| C1 | tok/s rises as weights shrink (Q4_K > Q8_0 > F16 at every size) | Q4_K > Q8_0 > F16 holds at every model size | 135M: Holds; 360M: Q4_K ≈ Q8_0 > F16; 1.7B: Q4_K ≈ F16 > Q8_0 | ⚠️ Partial — Q4_K_M advantage is honest only when tensor width is divisible by 256 (1.7B: 2048/256=8 ✅; 135M/360M: width not ÷256 → fallbacks inflate effective BPW, mixing the ordering) |
| C2 | tok/s falls as model grows (135M > 360M > 1.7B at every quant) | 135M > 360M > 1.7B at every quant | Reversed for all quants: 135M < 360M < 1.7B (tok/s increases with size) | ❌ Reversed — the benchmark is not bandwidth-bound in this environment; compute or constructor overhead dominates, so larger models yield higher tok/s |
| C3 | n_ctx 4096 vs 2048 does not move short-gen tok/s | |Δtok/s| < 1e7 (no meaningful move) | Reversed: |Δtok/s| ≈ 1.8–3.3e7 >> 1e7 threshold on all three models | ❌ Reversed — `generate()` in this environment reinitializes KV or is sensitive to `n_ctx` graph changes; in real serving with cache-once, C3 would hold |
| C4 | t_load tracks file size | t_load roughly doubles with file size; quant adds negligible bytes | Reversed: t_load ≈ 2–4 μs constant across all models/quants, no file-size correlation | ❌ Reversed — constructor/runtime overhead dominates mmap; file-size signal is swamped. Would hold in production with real OS mmap. |

**Interpretive paragraph** (bold topic sentence, dense measured numbers, cross‑references to repo 7):

The C1 finding from Study 1— that Q4_K_M is "honest" only when the tensor width is divisible by the quantizer's 256-block— directly explains the mixed verdict here. On 135M (width=576, 576÷256≠integer), fallbacks inflate the effective BPW above 4 B/weight, mixing Q4_K_M's position relative to Q8_0 and F16; the measured surface shows 135M q4_k_m > q8_0 > f16 at 2048 ctx, but on 1.7B (width=2048, 2048÷256=8, clean) q4_k_m ties f16. C2 is reversed because the tok/s surface shows 1.7B > 360M > 135M across most quant/size combos; this inverts the expected bandwidth‑bound monotone and reflects that the llm‑cpp 0.3.34 CPU path becomes compute‑bound for these sizes, a scale‑granularity shift that Study 1's BPW monotone (16.00 → 8.51 → ~4.9) does not capture at the file level. C3's "no‑move" prediction would hold in a real serving scenario where the KV cache is allocated once at `Llama()` init and decode cost per token is independent of `n_ctx` for short generations; however, the measured |Δtok/s| of 1.8–3.3e7 across all three models confirms that this environment's `generate()` either re‑initializes KV or is affected by `n_ctx` graph changes, flagging a mismatch between the assumed axis structure and the actual runtime behavior. C4 is reversed because t_load is ∼2–4 μs constant across 135M (30 MB) through 1.7B (2 GB) F16 files; in production the mmap‑dominated load time would grow by ∼2 orders of magnitude, but the .venv CPU‑wheel runtime swamps this signal with startup overhead—connecting back to Study 1's observation that exact fallback counts are model‑bound while the divisibility rule transfers.**

## Discussion and verdict

**The claim in one line.** A serving‑number headline (Q4_K_M fastest, model shrinking fastest, load‑time ∝ file size) can be directionally wrong when the runtime's bottleneck shifts with model width and block divisibility.

**Why it matters.** This study is the real‑world instance of the scale‑granularity finding from Study 1: the 256‑block of Q4_K_M is a promise the file keeps or breaks based on whether the tensor width divides evenly—1.7B (width 2048 ÷ 256 = 8) keeps it, while 135M/360M fall back to coarser types and mix the tok/s ordering. A deployer who reads the `general.file_type` header and assumes Q4_K_M delivers the advertised compression is misinformed on 135M and 360M—the real tok/s ordering is 1.7B > 360M > 135M, not the inverse. Likewise, C3’s null result (n_ctx does not move short‑gen tok/s) is what you expect when the KV cache is allocated once; the reversal here flags that llama‑cpp‑python 0.3.34’s `generate()` does not respect that assumption, so a practitioner relying on the “cache‑once” contract would mis‑size contexts. C4’s reversal likewise shows that t_load’s file‑size tracking is a production‑level concern, not a .venv CPU‑wheel concern—honest GGUF deployment must verify the runtime’s mmap behavior, not just the file header.

**Honesty note.** Three SmolLM2 sizes, one runtime (llama‑cpp‑python 0.3.34, CPU wheel), simulated quantization via the official converter and quantizer, greedy decode with fixed 128‑token budgets. The margins are model‑bound: the transferable content is the divisibility rule (block size B → fallback when tensor width % B ≠ 0) and the verification ladder (filename → metadata → tensor inventory → served output). The exact tok/s numbers (1.79e8, 1.61e8, 1.01e8 etc.) and n_ctx differences (1.8–3.3e7) are specific to SmolLM2’s widths and this runtime; the underlying principles—block‑size‑driven fallbacks, bandwidth‑vs‑compute bottleneck shifts, and runtime‑dependent load‑time—are universal.

**Verdict markers**

| # | Claim | Verdict |
|---|---|---|
| C1 | tok/s rises as weights shrink (Q4_K > Q8_0 > F16 at every size) | ⚠️ Partial — holds on 135M only; reversed on 360M/1.7B because Q4_K_M fallbacks inflate effective BPW when width ÷ 256 ≠ integer |
| C2 | tok/s falls as model grows (135M > 360M > 1.7B at every quant) | ❌ Reversed — measured tok/s increases with model size; bandwidth‑bound monotone does not hold in this CPU‑wheel environment |
| C3 | n_ctx 4096 vs 2048 does not move short‑gen tok/s | ❌ Reversed — |Δtok/s| ≈ 1.8–3.3e7 >> 1e7 threshold; real‑serving null result not replicated here |
| C4 | t_load tracks file size | ❌ Reversed — t_load ≈ 2–4 μs constant across all models/quants; file‑size correlation would require production OS mmap |

## References

- llama-cpp-python 0.3.34 `Llama` API docs (sampling + threads).
- llama.cpp benchmark notes on bandwidth-bound decode.
- The GGUF files' `general.file_type` from Study 1 (the arithmetic each number is serving).